<a href="https://colab.research.google.com/github/tougheye/Data_processing/blob/main/CES_membership_upte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This project is to identify HX workers (names and titles) have departmental overlap with Community Educ Spec titles in Unit 99

In [ ]:
import pandas as pd

upte_db_file_path = "/content/drive/MyDrive/Data/UPTE_Database_Employee_Outputs/upte_db_ee_all_apt_20260902.xlsx"
upte_db_df = pd.read_excel(upte_db_file_path)
upte_db_df.head()

In [13]:
upte_db_df.shape

(116560, 62)

In [10]:
upte_db_df['prSource'].unique()

array(['ucsc', 'uci', 'ucsb', 'ucr', 'ucla', 'ucb', 'ucsd', 'lbnl',
       'ucsf', 'ucd', nan, 'lanl', 'llnl', 'ucm', 'ucop', 'danr'],
      dtype=object)

Need to dedupe the upte_db_df based on upteID, titleCode, campusID to make the list of unique employee and job titles

In [14]:
upte_db_unique_title_campus_df = upte_db_df.drop_duplicates(subset=['upteID', 'titleCode', 'campusID'], inplace=True)
upte_db_unique_title_campus_df.shape

(109149, 62)

In [11]:
# The upteID column came up twice in the original import

upte_db_df.drop(columns=['upteID.1'], inplace=True)
# upte_db_df.head()

Standardize the fnames, lnames, and email addresses to match against statewide CES list

In [12]:
upte_db_df['firstName'] = upte_db_df['firstName'].str.strip().str.lower()
upte_db_df['lastName'] = upte_db_df['lastName'].str.strip().str.lower()
upte_db_df['email'] = upte_db_df['email'].str.strip().str.lower()
upte_db_df['altEmail'] = upte_db_df['altEmail'].str.strip().str.lower()
upte_db_df.head()

,appointmentID,apptNum,upteID,campusID,titleCode,apptPercentage,isPrimaryAppointment,lastModDate,Occur_cnt,unique_title_cnt,...,isCardCollected,isCardCollectedDate,isActive,isActiveChangeDate,prSource,trackerID,businessUnit,lastPayDate,lastDuesPayDate,lastModDate.1
0,2167317,2,10,7.0,9611,1.0,y,2025-09-12 07:00:00,2,1,...,n,NaT,n,2025-09-12 00:00:00,ucsc,8.0,NaN,2025-09-01,2025-09-01,2026-02-06 11:02:35
1,2105671,1,10,7.0,9611,1.0,y,2024-07-03 07:48:10,2,1,...,n,NaT,n,2025-09-12 00:00:00,ucsc,8.0,NaN,2025-09-01,2025-09-01,2026-02-06 11:02:35
2,2109901,1,27,7.0,7359,1.0,y,2024-07-10 07:45:57,2,1,...,n,NaT,n,2024-10-20 00:00:00,ucsc,NaN,NaN,2024-10-12,2024-10-12,2025-09-18 01:19:08
3,2134099,2,27,7.0,7359,1.0,y,2024-10-20 07:00:00,2,1,...,n,NaT,n,2024-10-20 00:00:00,ucsc,NaN,NaN,2024-10-12,2024-10-12,2025-09-18 01:19:08
4,2105675,1,152,7.0,9723,1.0,y,2025-10-23 00:00:00,2,1,...,n,NaT,n,2026-01-08 00:00:00,ucsc,NaN,SCCMP,2026-01-03,2026-01-03,2026-02-18 00:00:00


In [9]:
statewide_ces_file = pd.ExcelFile("/content/Statewide CES List.xlsx")
ces_tabs = statewide_ces_file.sheet_names
print(ces_tabs)

['NEW 99', 'Pivot Table 3', 'Detail1-patient-facing-Nursing']


In [10]:
statewide_ces_df = statewide_ces_file.parse(ces_tabs[0])
statewide_ces_df.shape

(455, 28)

In [13]:
statewide_ces_df['First'] = statewide_ces_df['First'].str.strip().str.lower()
statewide_ces_df['Last Name'] = statewide_ces_df['Last Name'].str.strip().str.lower()
statewide_ces_df['EE Work Email Address'] = statewide_ces_df['EE Work Email Address'].str.strip().str.lower()
statewide_ces_df.head()

,Notes,Location/Business Unit,First,Last Name,Working Title,Department Description,Job Code Description,Job Code,Most Recent Date of Hire in Empl Rcd,Job Expected End Date,...,EE Work Location Address 1,EE Work Location Address 2,EE Work Location Address 3,EE Work Location Floor,EE Work Location City,EE Work Location State,EE Work Location Zip Code,EE Work Email Address,EE Work Phone,Employee Relation Description
0,patient-facing,DVCMP,karla,cuadros,NaN,PSYCHOLOGY,CMTY EDUC SPEC 1,5840,2008-07-31,NaT,...,200 East Quad,,,,Davis,CA,95616-5270,kacuadros@ucdavis.edu,530/754-4343,"All Others, Not Confidential"
1,patient-facing,DVCMP,steven,ruder,NaN,MED:MIND INSTITUTE,CMTY EDUC SPEC 4,5834,2013-10-16,NaT,...,2825 50th Street,,,,Sacramento,CA,95817,sruder@ucdavis.edu,916/703-0233,"All Others, Not Confidential"
2,patient-facing,DVCMP,maribel,hernandez,Community Outreach Specialist,MED:MIND INSTITUTE,CMTY EDUC SPEC 4,5834,2013-07-10,NaT,...,2825 50th Street,,,,Sacramento,CA,95817,belhernandez@ucdavis.edu,916/703-0233,"All Others, Not Confidential"
3,patient-facing,DVCMP,benita,shaw,NaN,MED:MIND INSTITUTE,CMTY EDUC SPEC 3,5838,2019-11-12,NaT,...,Remote,,,,,,00000,bjcshaw@ucdavis.edu,916/703-0336,"All Others, Not Confidential"
4,patient-facing,DVCMP,katharine,owens,NaN,MED:MIND INSTITUTE,CMTY EDUC SPEC 3,5838,2021-09-07,NaT,...,2825 50th Street,,,,Sacramento,CA,95817,khowens@ucdavis.edu,415/810-4987,"All Others, Not Confidential"


In [18]:
campus_conversion_dict = {'DVCMP': 'ucd', 'DVMED': 'ucd',
                          'IRCMP': 'uci', 'LACMP': 'ucla',
                          'RVCMP': 'ucr', 'SDCMP': 'ucsd', 'SDMED': 'ucsd',
                          'SFCMP': 'ucsf', 'BKCMP': 'ucb',
                          'MECMP': 'ucm', 'SCCMP': 'ucsc',
                          'UCANR': 'ucd'}

In [19]:
statewide_ces_df['campus'] = statewide_ces_df['Location/Business Unit'].map(campus_conversion_dict)
statewide_ces_df.head()

,Notes,Location/Business Unit,First,Last Name,Working Title,Department Description,Job Code Description,Job Code,Most Recent Date of Hire in Empl Rcd,Job Expected End Date,...,EE Work Location Address 2,EE Work Location Address 3,EE Work Location Floor,EE Work Location City,EE Work Location State,EE Work Location Zip Code,EE Work Email Address,EE Work Phone,Employee Relation Description,campus
0,patient-facing,DVCMP,karla,cuadros,NaN,PSYCHOLOGY,CMTY EDUC SPEC 1,5840,2008-07-31,NaT,...,,,,Davis,CA,95616-5270,kacuadros@ucdavis.edu,530/754-4343,"All Others, Not Confidential",ucd
1,patient-facing,DVCMP,steven,ruder,NaN,MED:MIND INSTITUTE,CMTY EDUC SPEC 4,5834,2013-10-16,NaT,...,,,,Sacramento,CA,95817,sruder@ucdavis.edu,916/703-0233,"All Others, Not Confidential",ucd
2,patient-facing,DVCMP,maribel,hernandez,Community Outreach Specialist,MED:MIND INSTITUTE,CMTY EDUC SPEC 4,5834,2013-07-10,NaT,...,,,,Sacramento,CA,95817,belhernandez@ucdavis.edu,916/703-0233,"All Others, Not Confidential",ucd
3,patient-facing,DVCMP,benita,shaw,NaN,MED:MIND INSTITUTE,CMTY EDUC SPEC 3,5838,2019-11-12,NaT,...,,,,,,00000,bjcshaw@ucdavis.edu,916/703-0336,"All Others, Not Confidential",ucd
4,patient-facing,DVCMP,katharine,owens,NaN,MED:MIND INSTITUTE,CMTY EDUC SPEC 3,5838,2021-09-07,NaT,...,,,,Sacramento,CA,95817,khowens@ucdavis.edu,415/810-4987,"All Others, Not Confidential",ucd


In [ ]:
db_cols_to_keep = ['employeeID', 'firstName', 'lastName', 'midName', 'upteID',\
                   'hireDate', 'email', 'altEmail', 'prSource', 'lastPayDate',
                   'lastDuesPayDate', 'lastModDate', ]

In [20]:
upte_db_df.columns

Index(['employeeID', 'firstName', 'lastName', 'midName', 'upteID', 'name',
       'prefName', 'memberStatus', 'ledByStaffLeader', 'unitRepID',
       'workplaceRepID', 'affirmForm', 'hireDate', 'deptID', 'contactZoneID',
       'buildingID', 'ucWorkBuilding', 'altLocation', 'room', 'mailCode',
       'email', 'altEmail', 'isOKtoEmail', 'isOKtoText', 'phonePref',
       'createDate', 'lastModUser', 'lastlistDate', 'memberJoinDate',
       'supervisor', 'supervisorsName', 'flsaStatus', 'hrCenter', 'org',
       'campaignAssessCode', 'assessCode', 'assessDate', 'assessorID', 'orgID',
       'generalNote', 'ucWorkLoc', 'isCardCollected', 'isCardCollectedDate',
       'isActive', 'isActiveChangeDate', 'prSource', 'trackerID',
       'businessUnit', 'lastPayDate', 'lastDuesPayDate', 'lastModDate'],
      dtype='object')

In [ ]:
statewide_ces_df